In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

# ===================== 1. 配置路径和参数 =====================
# 你的数据集路径
DATA_FOLDER = "./de-en"
DE_FILE = "europarl-v7.de-en.de"
EN_FILE = "europarl-v7.de-en.en"

# 预训练模型（德英翻译专用，小而快）
MODEL_NAME = "Helsinki-NLP/opus-mt-de-en"

# 训练参数
MAX_SEQ_LEN = 32
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 2e-5

# ===================== 2. 读取本地数据 =====================
def read_data(de_path, en_path):
    with open(de_path, 'r', encoding='utf-8') as f:
        de_lines = [line.strip() for line in f]
    with open(en_path, 'r', encoding='utf-8') as f:
        en_lines = [line.strip() for line in f]
    
    # 确保句子数量一致
    assert len(de_lines) == len(en_lines), "德英句子数量不匹配！"
    return {"de": de_lines, "en": en_lines}

# 读取数据
data = read_data(
    os.path.join(DATA_FOLDER, DE_FILE),
    os.path.join(DATA_FOLDER, EN_FILE)
)

# 分别切片两个语言的句子列表
data["de"] = data["de"][:1000]
data["en"] = data["en"][:1000]

# 转换为Hugging Face Dataset格式，并切分训练/验证集
dataset = Dataset.from_dict(data)
dataset = dataset.train_test_split(test_size=0.1)  # 10%作为验证集
print(f"✅ 数据加载完成：训练集 {len(dataset['train'])} 条，验证集 {len(dataset['test'])} 条")

# ===================== 3. 加载预训练分词器和模型 =====================
# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 加载预训练翻译模型（德译英）
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print("✅ 预训练模型加载完成")

# ===================== 4. 数据预处理（分词） =====================
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["de"],
        text_target=examples["en"], # 核心替代
        max_length=MAX_SEQ_LEN,
        truncation=True
    )
    return model_inputs

# 批量预处理数据
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["de", "en"]  # 移除原始文本列，只保留分词后的id
)

# 数据整理器（自动处理padding）
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# ===================== 5. 配置训练参数 =====================
training_args = Seq2SeqTrainingArguments(
    output_dir="./translation_model",  # 模型保存路径
    evaluation_strategy="epoch",        # 每个epoch评估一次
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    save_total_limit=2,  # 只保存最新的2个模型
    predict_with_generate=True,  # 翻译任务需要生成
    logging_steps=100,
)

# ===================== 6. 初始化Trainer并训练 =====================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# 开始训练！
print("🚀 开始训练...")
trainer.train()

2026-03-18 11:51:00.679606: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-18 11:51:00.679788: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-18 11:51:00.686575: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


✅ 数据加载完成：训练集 900 条，验证集 100 条


/home/hexiaoya/miniconda3/envs/ai/lib/python3.10/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


✅ 预训练模型加载完成


Map:   0%|          | 0/900 [00:00<?, ? examples/s]

...

/home/hexiaoya/miniconda3/envs/ai/lib/python3.10/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


🚀 开始训练...


Epoch,Training Loss,Validation Loss
1,No log,1.449510
2,No log,1.435414
3,No log,1.433985


/home/hexiaoya/miniconda3/envs/ai/lib/python3.10/site-packages/transformers/modeling_utils.py:2618: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[58100]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=87, training_loss=1.5697870364134339, metrics={'train_runtime': 17.7575, 'train_samples_per_second': 152.048, 'train_steps_per_second': 4.899, 'total_flos': 22881396326400.0, 'train_loss': 1.5697870364134339, 'epoch': 3.0})

In [2]:
# ===================== 7. 测试模型（训练完成后） =====================
def translate(text):
    """输入德语，输出英语"""
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LEN)
    inputs = inputs.to(model.device)
    outputs = model.generate(**inputs, max_length=MAX_SEQ_LEN)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# 测试几个句子
test_sentences = [
    "Wiederaufnahme der Sitzungsperiode",
    "Ich erkläre die Sitzungsperiode für wiederaufgenommen.",
    "Frau Präsidentin, zur Geschäftsordnung.",
    "Die Fähigkeit, komplexe Probleme systematisch zu analysieren und praxisnahe Lösungen zu entwickeln, ist eine der wichtigsten Kompetenzen in der modernen Arbeitswelt."
]

print("\n📝 测试翻译结果：")
for de in test_sentences:
    en = translate(de)
    print(f"德语：{de}")
    print(f"英语：{en}\n")


📝 测试翻译结果：
德语：Wiederaufnahme der Sitzungsperiode
英语：Resumption of the session

德语：Ich erkläre die Sitzungsperiode für wiederaufgenommen.
英语：I declare resumed the session of the European Parliament.

德语：Frau Präsidentin, zur Geschäftsordnung.
英语：Madam President, on a point of order.

德语：Die Fähigkeit, komplexe Probleme systematisch zu analysieren und praxisnahe Lösungen zu entwickeln, ist eine der wichtigsten Kompetenzen in der modernen Arbeitswelt.
英语：The ability to systematically analyse complex problems and develop practical solutions is one of the most important competences in the modern world of work.



In [3]:
import torch
print("PyTorch版本:", torch.__version__)
print("CUDA是否可用:", torch.cuda.is_available())
print("CUDA版本:", torch.version.cuda)

PyTorch版本: 2.4.0+cu121
CUDA是否可用: True
CUDA版本: 12.1
